In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset

class AircraftDataset(Dataset):
    def __init__(self, images_dir, label_file, class_to_idx, transform=None):
        self.images_dir = images_dir
        self.transform = transform
        self.samples = []

        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                img_id, family = parts[0], parts[1]
                img_path = os.path.join(images_dir, img_id + ".jpg")
                label = class_to_idx[family]
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [2]:
def build_class_index(label_file):
    families = set()
    with open(label_file, "r") as f:
        for line in f:
            _, fam = line.strip().split(" ", 1)
            families.add(fam)
    families = sorted(list(families))
    class_to_idx = {fam: i for i, fam in enumerate(families)}
    return class_to_idx, families


In [3]:
import torch
from torchvision import transforms

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [4]:
from google.colab import drive
drive.mount('/content/drive')

images_dir = "/content/drive/Shared drives/ECI271 Image Data/data/images"
train_file = "/content/drive/Shared drives/ECI271 Image Data/data/images_family_train.txt"
val_file = "/content/drive/Shared drives/ECI271 Image Data/data/images_family_val.txt"
test_file = "/content/drive/Shared drives/ECI271 Image Data/data/images_family_test.txt"

class_to_idx, families = build_class_index(train_file)

train_ds = AircraftDataset(images_dir, train_file, class_to_idx, train_tf)
val_ds   = AircraftDataset(images_dir, val_file, class_to_idx, test_tf)
test_ds  = AircraftDataset(images_dir, test_file, class_to_idx, test_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds, batch_size=32)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=32)


Mounted at /content/drive


In [5]:
!ls "/content/drive/Shared drives/ECI271 Image Data/data"





families.txt			  images_manufacturer_val.txt
images				  images_test.txt
images_box.txt			  images_train.txt
images_family_test.txt		  images_val.txt
images_family_train.txt		  images_variant_test.txt
images_family_trainval.txt	  images_variant_train.txt
images_family_val.txt		  images_variant_trainval.txt
images_manufacturer_test.txt	  images_variant_val.txt
images_manufacturer_train.txt	  manufacturers.txt
images_manufacturer_trainval.txt  variants.txt


In [6]:
import torch
import torch.nn as nn
from torchvision.models import resnet50

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = len(families)

model = resnet50(weights="IMAGENET1K_V2")
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 204MB/s]


In [7]:
# def train_epoch(model, loader, optimizer, criterion):
#     model.train()
#     total_loss = 0
#     correct = 0

#     for imgs, labels in loader:
#         imgs, labels = imgs.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(imgs)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         correct += (outputs.argmax(1) == labels).sum().item()

#     return total_loss / len(loader), correct / len(loader.dataset)


# def eval_epoch(model, loader, criterion):
#     model.eval()
#     total_loss = 0
#     correct = 0

#     with torch.no_grad():
#         for imgs, labels in loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             outputs = model(imgs)
#             loss = criterion(outputs, labels)

#             total_loss += loss.item()
#             correct += (outputs.argmax(1) == labels).sum().item()

#     return total_loss / len(loader), correct / len(loader.dataset)

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct = 0, 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()

        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    avg_acc = total_correct / len(loader.dataset)
    return avg_loss, avg_acc


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct = 0, 0

    pbar = tqdm(loader, desc="Validating", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            total_correct += (outputs.argmax(1) == labels).sum().item()

            pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader.dataset)
    avg_acc = total_correct / len(loader.dataset)
    return avg_loss, avg_acc




In [8]:
'mode' in globals()

False

In [9]:
epochs = 10

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")


Epoch 1/10
  Train Loss: 3.7535 | Train Acc: 0.1137
  Val Loss:   3.1018 | Val Acc:   0.2535


Epoch 2/10
  Train Loss: 2.3382 | Train Acc: 0.4163
  Val Loss:   1.6461 | Val Acc:   0.5497


Epoch 3/10
  Train Loss: 1.0738 | Train Acc: 0.7522
  Val Loss:   0.9957 | Val Acc:   0.7327


Epoch 4/10
  Train Loss: 0.4722 | Train Acc: 0.9145
  Val Loss:   0.7252 | Val Acc:   0.7951


Epoch 5/10
  Train Loss: 0.2212 | Train Acc: 0.9658
  Val Loss:   0.6659 | Val Acc:   0.8143


Epoch 6/10
  Train Loss: 0.1081 | Train Acc: 0.9862
  Val Loss:   0.6255 | Val Acc:   0.8257


Epoch 7/10
  Train Loss: 0.0594 | Train Acc: 0.9955
  Val Loss:   0.6225 | Val Acc:   0.8224


Epoch 8/10
  Train Loss: 0.0388 | Train Acc: 0.9973
  Val Loss:   0.6162 | Val Acc:   0.8242


Epoch 9/10
  Train Loss: 0.0340 | Train Acc: 0.9955
  Val Loss:   0.6127 | Val Acc:   0.8248


Epoch 10/10
  Train Loss: 0.0346 | Train Acc: 0.9967
  Val Loss:   0.5882 | Val Acc:   0.8425


In [10]:
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"Test Accuracy: {test_acc:.4f}")


Test Accuracy: 0.8398
